# Chapter 5: Generating Test Questions from Geometric Structure

**Learning objectives:**
- Understand the eight geometric generators and how each exercises a different GMS primitive
- Generate questions with provable ground truth and verify them against the store
- See how the bridge layer converts eval questions to DOE format
- Use DOE expansion to multiply base questions into factor-varied test cases

> **LLM API key required.** This notebook makes real LLM calls
> (e.g. `anthropic.Anthropic()`). The provider SDK reads the key
> from the process environment, not from `.env` — so either
> `source .env` before launching Jupyter, or export
> `ANTHROPIC_API_KEY` (or whichever provider key matches your
> `GMS_LLM_MODEL`) in the shell that runs the kernel.
>
> See `.env.example` at the repo root for the full list of
> supported provider env vars, and `docs/internal_team_install.md`
> §3B › "LLM API key setup" for the step-by-step recipe.


---
## Preamble

This notebook builds on the GMS expert store created in Chapter 4. Instead of querying the store manually, we use **question generators** — classes that mine the geometric structure to produce test questions with provable ground truth.

The key layers involved:
- `harness.doe.generators` — eight geometric generators, each exercising a different GMS primitive
- `harness.doe.generators_taxonomy` — five taxonomy generators for adversarial, boundary and robustness testing
- `harness.doe.bridge` — adapter that converts eval questions to DOE format
- `graphdoe` — the DOE expansion engine (DesignMatrix, QuestionRephraser)

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store (no retraining — loads from disk)
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="gmsh_testing_ext/lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()

stats = store.stats()
print(f"\nStore: {stats['entities']} entities, {stats['relations']} relations, "
      f"{stats['triples']} triples, {stats['enm_entries']} ENM")

---

## The Generator Architecture

Every question generator follows the same protocol:

1. **Mine** the store for candidate facts (triples, ENM entries, paths).
2. **Construct** a natural-language question whose answer is uniquely determined by the store.
3. **Attach** a `gms_answer_fn(store)` closure that recomputes the answer at verification time.

The base class `QuestionGenerator` defines this contract:

```python
class QuestionGenerator(ABC):
    category: str = ""

    def generate(self, store) -> list[GeneratedQuestion]: ...
    def sample(self, candidates, max_questions, seed=42): ...
    def _validate(self, store, question) -> bool: ...
```

Each generator produces `GeneratedQuestion` objects with six fields: `qid`, `category`, `natural_language`, `gms_answer_fn`, `ground_truth` and `answer_type`. The `_validate` method re-runs the answer function and checks that it matches the stored ground truth — a self-consistency check that catches generator bugs before they reach the evaluation pipeline.

The eight **geometric generators** each exercise a different GMS primitive:

| Generator | GMS Primitive | Question Form | Answer Type |
|-----------|--------------|---------------|-------------|
| `PlausibilityGenerator` | `score_triple()` | "Which is more plausible: A or B?" | `str` (A/B) |
| `TensionGenerator` | `tension_energy()` | "Do these entities agree, contradict or are unrelated?" | `str` |
| `HolonomyGenerator` | `check_holonomy()` | "Is the indirect path consistent?" | `bool` |
| `LinkPredictionGenerator` | `link_predict()` | "What is the X of Y?" | `str` |
| `MultiHopTransportGenerator` | `path_score()` | "Follow relation chain: what do you reach?" | `str` |
| `PhaseComparisonGenerator` | ENM `lookup_enm()` | "Which has higher value: A or B?" | `str` (A/B) |
| `ConditionalGenerator` | `score_triple()` cross-relation | "If X has P, what is its Q?" | `str` |
| `ProceduralGenerator` | `path_score()` composed | "Follow steps 1–3; list entities" | `set_str` |

### Running All Eight Geometric Generators

Let us instantiate all eight and run them against the store. The `geometric_generators()` registry function returns one instance of each.

In [ ]:
from knowlytix.harness.doe.generators import geometric_generators

geo_gens = geometric_generators()

print(f"{'Generator':30s} {'Category':22s} {'Generated':>10s} {'Valid':>8s}")
print("-" * 74)
geo_totals = {}
for gen in geo_gens:
    questions = gen.generate(store)
    valid = [q for q in questions if gen._validate(store, q)]
    geo_totals[gen.category] = (len(questions), len(valid))
    print(f"{type(gen).__name__:30s} {gen.category:22s} {len(questions):>10d} {len(valid):>8d}")

total = sum(v[0] for v in geo_totals.values())
print(f"\n{'TOTAL':30s} {'':22s} {total:>10d}")

From the 807 triples and 261 ENM entries in our store the eight geometric generators produce 334 questions — all 334 pass self-validation (the answer function reproduces the stored ground truth). The generators are deterministic: given the same store and seed they always produce the same questions.

Note that each generator produces a different number of candidates. PlausibilityGenerator mines 50 triples and filters for clear geodesic separation, yielding 76. PhaseComparisonGenerator draws 40 pairs from ENM entries. The smallest pool (18) comes from ConditionalGenerator because it requires entities with both a condition-type and consequence-type relation.

---

## Geometric Generators in Detail

Let us inspect sample questions from each generator to see how GMS primitives become test questions.

### PlausibilityGenerator — Geodesic Distance on $S^{m-1}$

This generator picks a true triple $(h, r, t)$ and a random wrong tail $t'$, then asks which tail is more plausible. The ground truth comes from comparing geodesic distances: $d(R_r(v_h), v_t) < d(R_r(v_h), v_{t'})$. Only pairs with a clear margin (>0.1 radians) are kept.

In [ ]:
from knowlytix.harness.doe.generators import PlausibilityGenerator

plaus_gen = PlausibilityGenerator()
plaus_qs = plaus_gen.generate(store)
sample = plaus_gen.sample(plaus_qs, 3, seed=42)

print(f"PlausibilityGenerator: {len(plaus_qs)} questions\n")
for q in sample:
    print(f"[{q.qid}]")
    print(f"  Q: {q.natural_language}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    meta = q.metadata
    if 'geodesic_real' in meta:
        print(f"  Geodesic (real): {meta['geodesic_real']:.4f}, "
              f"(wrong): {meta['geodesic_wrong']:.4f}, "
              f"margin: {meta['geodesic_wrong'] - meta['geodesic_real']:.4f}")
    else:
        print(f"  Metadata keys: {list(meta.keys())}")
    print()

The `in_section` triples have the widest margins (close to 1.0 radian) because section membership is the most consistent geometric signal. Relation-specific triples like `has_tier` have narrower margins (~0.2) but are still clearly separated. The metadata records both geodesic distances so we can audit the generator's decisions.

### TensionGenerator — Tension Energy

This generator picks entity pairs and classifies their relationship using tension energy thresholds:
- $E(a,b) < 0.8$ → **agree** (entities share significant geometric structure)
- $0.8 \leq E(a,b) \leq 1.6$ → **unrelated** (typical for entities in different sections)
- $E(a,b) > 1.6$ → **contradict** (opposing semantic content)

In [ ]:
from knowlytix.harness.doe.generators import TensionGenerator

tens_gen = TensionGenerator()
tens_qs = tens_gen.generate(store)
sample = tens_gen.sample(tens_qs, 3, seed=42)

print(f"TensionGenerator: {len(tens_qs)} questions\n")
for q in sample:
    print(f"[{q.qid}]")
    print(f"  Q: {q.natural_language[:120]}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    print(f"  Tension energy: {q.metadata['tension_energy']:.4f}")
    print()

Most entity pairs in this document fall in the "unrelated" band (tension energy 1.0–1.4). This makes sense: the model risk assessment describes 12 independent models, so cross-model entity pairs are geometrically distant but not contradictory. True contradiction requires entities with directly opposing semantics (e.g. "passes" vs. "fails").

### HolonomyGenerator — Operator Composition Consistency

Holonomy checks whether following two successive relation operators (via an intermediate entity) is consistent with the direct operator. Given a path $h \xrightarrow{r_1} m \xrightarrow{r_2} t$ and a direct relation $h \xrightarrow{r_d} t$, the holonomy defect measures $\| R_{r_2} \circ R_{r_1} - R_{r_d} \|$. A defect near zero means the indirect path is geometrically consistent.

In [ ]:
from knowlytix.harness.doe.generators import HolonomyGenerator

holo_gen = HolonomyGenerator()
holo_qs = holo_gen.generate(store)
sample = holo_gen.sample(holo_qs, 3, seed=42)

print(f"HolonomyGenerator: {len(holo_qs)} questions\n")
for q in sample:
    m = q.metadata
    print(f"[{q.qid}]")
    print(f"  Path: {m['head']} --[{m['r1']}]--> {m['mid']} --[{m['r2']}]--> {m['tail']}")
    print(f"  Direct relation: {m['r_direct']}")
    print(f"  Holonomy defect: {m['holonomy_defect']:.4f}")
    print(f"  Consistent? {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:   {computed}  (match={str(computed) == str(q.ground_truth)})")
    print()

The holonomy defects range from 0.5 to 1.3 — all exceeding the consistency threshold, so all these indirect paths are flagged as inconsistent. This is expected: the `has_downstream_model` relation connects different models, so composing it with `in_section` or `has_risk_tier` does not reproduce the direct relation's operator. Holonomy consistency is a strong signal for structural integrity in the knowledge graph.

### LinkPredictionGenerator — Geometric Beam Search

This generator asks "What is the X of Y?" and verifies the answer using the GMS link prediction engine, which ranks candidate tails by geodesic proximity to the operator-transported head $R_r(v_h)$.

In [ ]:
from knowlytix.harness.doe.generators import LinkPredictionGenerator

link_gen = LinkPredictionGenerator()
link_qs = link_gen.generate(store)
sample = link_gen.sample(link_qs, 3, seed=42)

print(f"LinkPredictionGenerator: {len(link_qs)} questions\n")
for q in sample:
    m = q.metadata
    print(f"[{q.qid}]")
    print(f"  Q: {q.natural_language}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    # Show top-3 predictions with geodesic scores
    print("  Top predictions:")
    for entity, score in m["top_predictions"]:
        marker = " <-- correct" if entity == q.ground_truth else ""
        print(f"    {entity:30s}  geodesic={score:.4f}{marker}")
    print()

The top-3 predictions show the geometric ranking: the correct tail often ranks 2nd or 3rd because the self-entity (MDL-00X) is always closest geometrically to its own operator-transported position. The generator filters for questions where the correct answer appears in the top-3 candidates.

### MultiHopTransportGenerator — Two-Hop Relational Paths

This generator constructs two-hop queries: "Starting from entity A, follow relation $r_1$ to reach B, then follow $r_2$. What do you reach?" The path geodesic score measures how well the composed transport $R_{r_2}(R_{r_1}(v_A))$ matches the final entity.

In [ ]:
from knowlytix.harness.doe.generators import MultiHopTransportGenerator

mhop_gen = MultiHopTransportGenerator()
mhop_qs = mhop_gen.generate(store)
sample = mhop_gen.sample(mhop_qs, 3, seed=42)

print(f"MultiHopTransportGenerator: {len(mhop_qs)} questions\n")
for q in sample:
    m = q.metadata
    print(f"[{q.qid}]")
    print(f"  Path: {m['head']} --[{m['r1']}]--> {m['mid']} --[{m['r2']}]--> {m['tail']}")
    print(f"  Q: {q.natural_language[:120]}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    print(f"  Path geodesic: {m['path_geodesic']:.4f}")
    print()

The multi-hop paths typically go through model entities: "Starting from an action item, follow `has_model_id` to MDL-001, then follow `has_approval_rate`." Path geodesic scores of 1.3–1.7 indicate moderate geometric alignment — the composed operator transport is approximate but sufficient for the ground truth to be uniquely recoverable.

### PhaseComparisonGenerator — ENM Numeric Comparisons

This generator draws two ENM entries from the same section and asks which has the higher value. The ground truth comes from the exact IEEE 754 values in the ENM register — no geometric approximation involved.

In [ ]:
from knowlytix.harness.doe.generators import PhaseComparisonGenerator

phase_gen = PhaseComparisonGenerator()
phase_qs = phase_gen.generate(store)
sample = phase_gen.sample(phase_qs, 3, seed=42)

print(f"PhaseComparisonGenerator: {len(phase_qs)} questions\n")
for q in sample:
    m = q.metadata
    print(f"[{q.qid}]")
    print(f"  Q: {q.natural_language[:120]}")
    print(f"  Ground truth: {q.ground_truth}")
    computed = q.gms_answer_fn(store)
    print(f"  Verified:     {computed}  (match={computed == q.ground_truth})")
    print(f"  ENM section: {m['category']}")
    print(f"  Value A: {m['value_a']:.4f} ({m['entity_a'][:40]})")
    print(f"  Value B: {m['value_b']:.4f} ({m['entity_b'][:40]})")
    print()

The PhaseComparisonGenerator is the only geometric generator that uses the ENM register rather than the hypersphere embeddings. This makes it complementary to the others: it tests whether an agent can reason about exact numeric values, while the others test geometric understanding.

### ConditionalGenerator — Cross-Relation Rotor Test

Many real-world queries have an if-then structure: "If this model's risk tier is Critical, what remediation deadline applies?" Geometrically, a conditional is a *cross-relation rotor test*. For an entity $e$ the knowledge graph contains two distinct relations — the condition $(e, r_\text{cond}, t_\text{cond})$ and the consequence $(e, r_\text{cons}, t_\text{cons})$. The ground truth is verified by applying the consequence rotor: $d(R_{r_\text{cons}}(v_e), v_{t_\text{cons}}) < \tau$.

In [ ]:
from knowlytix.harness.doe.generators import ConditionalGenerator

cond_gen = ConditionalGenerator()
cond_qs = cond_gen.generate(store)
sample = cond_gen.sample(cond_qs, 3, seed=42)

print(f"ConditionalGenerator: {len(cond_qs)} questions\n")
for q in sample:
    m = q.metadata
    ctype = m.get('conditional_type', 'single')
    print(f"[{q.qid}] type={ctype}")
    print(f"  Entity: {m['entity']}")
    if ctype == 'single':
        print(f"  IF   {m['condition_relation']} = {m['condition_value']}")
        print(f"  THEN {m['consequence_relation']} = {m['consequence_value']}")
    else:
        print(f"  IF   {m['condition_1']} AND {m['condition_2']}")
        print(f"  THEN {m['consequence_relation']} = {m['consequence_value']}")
    computed = q.gms_answer_fn(store)
    print(f"  Geodesic: {m['geodesic']:.4f}  Verified: {computed == q.ground_truth}")
    print()

Geodesic scores below 1.0 confirm that the consequence rotor maps the entity embedding close to the expected tail — the conditional relationship is geometrically well-grounded. The generator supports both single-condition ("if P then Q") and compound-condition ("if P and Q then R") forms.

### ProceduralGenerator — Ordered Dependency Chains

Agents operating on structured documents must often follow ordered procedures: trace a dependency chain, walk a workflow sequence, or enumerate the steps of a remediation plan. The ProceduralGenerator tests this by asking the agent to follow a multi-step chain and report the entities visited in sequence.

The generator produces two types:
1. **Same-relation chains** — follow a single relation type repeatedly (e.g., `has_downstream_model`) to produce a dependency cascade.
2. **Mixed-relation workflows** — follow a heterogeneous 3-hop chain where each step uses a different relation.

Ground truth is the ordered entity list, verified by composed rotor transport: $\text{path\_score}(v_{e_0}, [r_1, r_2, r_3], v_{e_3})$.

In [ ]:
from knowlytix.harness.doe.generators import ProceduralGenerator

proc_gen = ProceduralGenerator()
proc_qs = proc_gen.generate(store)
sample = proc_gen.sample(proc_qs, 3, seed=42)

print(f"ProceduralGenerator: {len(proc_qs)} questions\n")
for q in sample:
    m = q.metadata
    ptype = m['procedural_type']
    print(f"[{q.qid}] type={ptype}")
    print(f"  Chain: {' -> '.join(m['chain'])}")
    if ptype == 'mixed_relation':
        steps_str = ' -> '.join(f'[{r}] {t}' for r, t in m['steps'])
        print(f"  Steps: {steps_str}")
    else:
        print(f"  Relation: {m['relation']}")
    computed = q.gms_answer_fn(store)
    print(f"  Path geodesic: {m['path_geodesic']:.4f}  Verified: {str(computed) == str(q.ground_truth)}")
    print()

Path geodesic scores of 1.4–1.7 indicate that the composed operator transport is approximate but sufficient for the ground truth to be uniquely recoverable. The ordered entity list provides a strict correctness criterion — partial or reordered answers are marked incorrect.

---

## Lab 5.1 — Generate and Inspect Test Questions

Let us now use the `geometric_generators()` registry to generate questions, verify them and examine the full question anatomy.

In [ ]:
# Lab 5.1: Generate, sample and verify geometric questions
from knowlytix.harness.doe.generators import geometric_generators

all_geo_qs = []
for gen in geometric_generators():
    questions = gen.generate(store)
    sampled = gen.sample(questions, max_questions=3, seed=42)
    # Verify each sampled question
    for q in sampled:
        computed = q.gms_answer_fn(store)
        assert str(computed) == str(q.ground_truth), \
            f"Verification failed for {q.qid}: {computed} != {q.ground_truth}"
    all_geo_qs.extend(sampled)

print(f"Sampled {len(all_geo_qs)} questions (3 per generator x 8 generators)")
print(f"\n{'QID':35s} {'Category':22s} {'Answer':>10s} {'Type':>6s}")
print("-" * 77)
for q in all_geo_qs:
    gt = str(q.ground_truth)[:10]
    print(f"{q.qid:35s} {q.category:22s} {gt:>10s} {q.answer_type:>6s}")

In [ ]:
# Anatomy of a single question
q = all_geo_qs[0]
print("=== Question Anatomy ===")
print(f"  qid:             {q.qid}")
print(f"  category:        {q.category}")
print(f"  answer_type:     {q.answer_type}")
print(f"  natural_language: {q.natural_language}")
print(f"  ground_truth:    {q.ground_truth}")
print(f"  llm_prompt:      {q.llm_prompt[:150]}...")
print(f"  gms_answer_fn:   {q.gms_answer_fn}")
print(f"  metadata keys:   {list(q.metadata.keys())}")
print("\nFull metadata:")
for k, v in q.metadata.items():
    print(f"    {k}: {v}")

Every question carries its full provenance: which entities are involved, the geodesic scores that determine the ground truth and the answer function closure that can recompute the answer at any time. This makes each question independently auditable — you can trace every ground truth back to a specific GMS primitive and verify it.

---

## Taxonomy Generators: Failure-Mode Testing

The eight geometric generators verify whether an agent *understands* the document. A separate family — the **taxonomy generators** — tests whether the agent *behaves safely* under stress. Each taxonomy generator injects a specific perturbation while preserving the ground truth:

| Generator | Category | What It Tests |
|-----------|----------|---------------|
| `BoundaryGenerator` | boundary | Answers at phase-encoder threshold limits |
| `AdversarialGenerator` | adversarial | Contradictory context hints and instruction injection |
| `RobustnessGenerator` | robustness | Typos, synonym substitution and entity truncation |
| `RecoveryTestGenerator` | recovery_test | Graceful degradation under simulated tool failures |
| `GovernanceTestGenerator` | governance | Forbidden-tool temptation, missing approvals, insufficient evidence |

Together the two families — geometric (8) and taxonomy (5) — form a **13-generator test suite**. Legacy "classic eval" generators (ExactRecall, Contradiction, MultiHop, etc.) are superseded by their geometric counterparts and excluded from the default pipeline.

In [ ]:
# Taxonomy generators
from knowlytix.harness.doe.generators_taxonomy import taxonomy_generators

tax_gens = taxonomy_generators()
print("=== Taxonomy Generators ===")
print(f"{'Generator':30s} {'Category':20s} {'Generated':>10s} {'Valid':>8s}")
print("-" * 72)
for gen in tax_gens:
    qs = gen.generate(store)
    valid = [q for q in qs if gen._validate(store, q)]
    print(f"{type(gen).__name__:30s} {gen.category:20s} {len(qs):>10d} {len(valid):>8d}")
    if valid:
        q = valid[0]
        print(f"  Sample: {q.natural_language[:90]}")
        print(f"  GT: {str(q.ground_truth)[:60]}")

All 171 taxonomy questions pass validation. The taxonomy generators inject adversarial perturbations — typos in entity names (Robustness), contradictory context hints (Adversarial), simulated tool failures (RecoveryTest), threshold edge cases (Boundary) and tool-restriction constraints (Governance) — while preserving the ground truth.

---

## The Bridge: From Generators to DOE Format

The generators produce `GeneratedQuestion` objects, each carrying an answer-verification closure that can recompute the ground truth from the GMS store at any time. The DOE expansion engine (`graphdoe`) expects a slightly different question format. The **bridge** module (`harness.doe.bridge`) adapts between the two by wrapping the verification closure so that the DOE machinery can treat GMS-backed questions transparently.

The `generate_base_questions()` function runs all 13 default generators, validates results and returns DOE-ready questions capped at a fixed number per category:

In [ ]:
# Bridge: generate all 13 default base questions (geometric + taxonomy)
from knowlytix.harness.doe.bridge import generate_base_questions

all_base = generate_base_questions(store, max_per_category=10, seed=42)

print(f"\nAll base questions: {len(all_base)}")
cat_counts = Counter(q.category for q in all_base)
print(f"\n{'Category':25s} {'Count':>6s}")
print("-" * 33)
for cat, cnt in sorted(cat_counts.items()):
    print(f"{cat:25s} {cnt:>6d}")
print(f"\n{'TOTAL':25s} {len(all_base):>6d} ({len(cat_counts)} categories)")

The bridge caps each category at 10 questions (via `max_per_category`), turning the 505 raw candidates into a balanced set of 130 base questions across 13 categories.

---

## The DOE Factor System

Before we expand base questions into DOE test cases, we need to understand the **factor system**. DOE factors are experimental variables that modify how a question is presented — without changing the ground truth. The `graphdoe.FactorLoader` manages a catalog of 40 factors organized into 10 families and 13 preset groups.

In [ ]:
from knowlytix.harness.graphdoe import FactorLoader

fl = FactorLoader()

# Factor families (10 families, 40 factors total)
families = fl.list_factors()
total_factors = sum(len(v) for v in families.values())
print(f"=== Factor Catalog: {total_factors} factors in {len(families)} families ===\n")
for family, names in families.items():
    print(f"  {family:25s} ({len(names):>2d}): {names}")

In [ ]:
# Preset groups
groups = sorted(fl.list_groups())
print(f"=== Preset Groups ({len(groups)}) ===\n")
for gn in groups:
    factors = fl.load_by_group(gn)
    names = [f["name"] for f in factors]
    print(f"  {gn:20s} ({len(factors):>2d} factors): {names}")

In [ ]:
# The quick_screen group: 3 factors, ideal for demos
qs_factors = fl.load_by_group("quick_screen")
print("=== quick_screen factors ===\n")
for f in qs_factors:
    print(f"  {f['name']:25s} ({f['cardinality']} levels): {f['categories']}")
    print(f"    {f['description']}")

The `quick_screen` group uses just three factors — clarity, entity aliasing and reasoning cue — each with three levels (27 possible combinations). This is enough to detect major sensitivity patterns without requiring thousands of test runs. In Chapter 3 we introduced the design matrix; now we will see it in action.

---

## Lab 5.2 — DOE Expansion

### The Design Matrix

A `DesignMatrix` generates a space-filling sample from the factor space using Sobol sequences. Three factors with three levels each produce a full factorial space of 3³ = 27 possible combinations. Evaluating all 27 for every base question would be expensive, so a Sobol design with `n_runs=4` selects four well-spaced combinations from the 27. The output is a table with four rows (one per selected combination) and three columns (one per factor):

In [ ]:
from knowlytix.harness.graphdoe import DesignMatrix

dm = DesignMatrix(qs_factors, n_runs=4, seed=42)
print(f"Design matrix: {dm.n_runs} runs x {len(qs_factors)} factors")
print(f"Method: {dm.method}\n")
print(dm.matrix.to_string())

Each row is a unique combination of factor levels. Row 3 (`Clear`, `Exact`, `Direct`) is the "clean" baseline — no ambiguity, no aliasing, straightforward prompting. Row 1 (`Misleading`, `Synonym`, `Step_By_Step`) is the most adversarial — the question text is deliberately misleading, entity names are paraphrased and a chain-of-thought cue is injected.

Each of the 130 base questions is rephrased under all four conditions, so the total DOE test set is 130 × 4 = 520 questions. Statistical replication comes from the questions themselves: all 130 questions tested under the same condition provide enough observations to estimate that condition's effect.

### Expanding Base Questions

The `QuestionRephraser` transforms each base question according to a factor assignment. It supports two methods:

1. **Template** (`method="template"`) — deterministic string transformations: prepending tags, lowercasing entity names, injecting chain-of-thought cues. Fast, reproducible and free.
2. **LLM** (`method="llm"`) — sends each question to an LLM that rewrites the natural language while preserving the ground truth. Produces more realistic adversarial phrasings at the cost of API calls and non-determinism.

Both methods preserve the critical invariant: the ground truth and verification closure are never modified. Only the question text changes.

#### Template Method

The template rephraser applies mechanical rules for each factor level:

In [ ]:
from knowlytix.harness.graphdoe import QuestionRephraser

# Template rephraser: deterministic string transforms
qr_template = QuestionRephraser(method="template")

# Expand 3 plausibility questions under 4 conditions
base_plaus = generate_base_questions(
    store, generator_names=["plausibility"], max_per_category=3, seed=42
)
doe_template = qr_template.expand(base_plaus, dm.matrix)

print(f"Template expansion: {len(base_plaus)} base x {dm.n_runs} conditions "
      f"= {len(doe_template)} DOE questions")

# Show one base question across all 4 conditions
base_id = base_plaus[0].qid
variants = [dq for dq in doe_template if dq.base_qid == base_id]

print(f"\n--- '{base_id}' across {len(variants)} conditions (template) ---")
print(f"  Original: {base_plaus[0].natural_language}\n")
for v in variants:
    fa = v.factor_assignment
    tag = "/".join(fa.values())
    print(f"  Condition {v.doe_qid[-3:]} ({tag}):")
    print(f"    {v.question.natural_language}")
    print(f"    GT: {v.question.ground_truth}")
    print()

The template rephraser applies simple rules:

- **Clarity=Ambiguous**: prepends an `[Ambiguous]` tag
- **Clarity=Misleading**: prepends "Some sources suggest a different figure, but..." and lowercases entity names
- **Entity_aliasing=Abbreviated**: truncates entity names to abbreviations
- **Entity_aliasing=Synonym**: lowercases and paraphrases entity names
- **Reasoning_cue=Step_By_Step**: injects a chain-of-thought cue
- **Clear/Exact/Direct**: leaves the question unchanged (baseline)

The ground truth remains identical across all four conditions. Template rephrasing is deterministic — running it twice with the same seed produces identical output. This makes it ideal for reproducible benchmarks and regression testing.

#### LLM Method

The LLM rephraser sends each question to an LLM with instructions for each factor level. The LLM rewrites the question naturally while preserving the answer. This produces more realistic adversarial variations — the kind of phrasing a real user might actually type.

In [ ]:
# LLM rephraser: natural-language rewriting via the provider-agnostic
# knowlytix.core.llm.LLMClient. The bound model is driven by GMS_LLM_MODEL
# (default resolves to anthropic/claude-sonnet-4-20250514 via
# LLMSettings.DEFAULT_MODEL); to switch providers, set GMS_LLM_MODEL +
# the matching API key in your shell (or .env + `source .env` before
# launching jupyter). See .env.example and docs/internal_team_install.md
# §3B for the key-setup recipe.
from knowlytix.core.llm import get_llm

qr_llm = QuestionRephraser(method="llm", llm=get_llm())

# Expand the same 3 base questions under the same 4 conditions
doe_llm = qr_llm.expand(base_plaus, dm.matrix)

# Compare template vs LLM for the same base question
base_id = base_plaus[0].qid
tmpl_variants = [dq for dq in doe_template if dq.base_qid == base_id]
llm_variants = [dq for dq in doe_llm if dq.base_qid == base_id]

print(f"--- '{base_id}' : template vs LLM ---")
print(f"  Original: {base_plaus[0].natural_language}\n")

for tv, lv in zip(tmpl_variants, llm_variants):
    fa = tv.factor_assignment
    tag = "/".join(fa.values())
    print(f"  Condition {tv.doe_qid[-3:]} ({tag}):")
    print(f"    Template: {tv.question.natural_language}")
    print(f"    LLM:      {lv.question.natural_language}")
    print(f"    GT match: template={tv.question.ground_truth == base_plaus[0].ground_truth}, "
          f"llm={lv.question.ground_truth == base_plaus[0].ground_truth}")
    print()


The LLM rephraser produces more natural adversarial variations. Where the template method prepends a literal `[Ambiguous]` tag, the LLM rewrites the question with genuinely ambiguous phrasing. Where the template lowercases entity names, the LLM substitutes plausible synonyms or descriptions.

**Trade-offs:**

| | Template | LLM |
|--|----------|-----|
| **Speed** | Instant (regex) | ~1s per question (API call) |
| **Cost** | Free | ~$0.001 per question |
| **Determinism** | Identical across runs | Non-deterministic |
| **Realism** | Mechanical artifacts (tags, lowercasing) | Natural human-like phrasing |
| **Regression testing** | Ideal (reproducible) | Requires pinned outputs |

**Recommendation:** Use template for initial development and regression suites where reproducibility matters. Use LLM for final evaluation rounds where you want to test against realistic adversarial queries. The ground truth is preserved by both methods, so results are directly comparable.

### Full DOE Expansion via DOEGMSBenchmark

The `DOEGMSBenchmark.generate_questions()` method runs the full pipeline: all generators → bridge → sampling → DOE expansion.

In [ ]:
# Full DOE expansion via the benchmark harness
doe_questions = bench.generate_questions()

print(f"DOE questions: {len(doe_questions)}")
doe_cats = Counter(dq.question.category for dq in doe_questions)
print(f"\n{'Category':25s} {'Base':>6s} {'x Runs':>7s} {'= DOE':>7s}")
print("-" * 48)
for cat, cnt in sorted(doe_cats.items()):
    base_cnt = cnt // config.n_runs
    print(f"{cat:25s} {base_cnt:>6d} {'x ' + str(config.n_runs):>7s} {cnt:>7d}")
print(f"\n{'TOTAL':25s} {len(doe_questions) // config.n_runs:>6d} {'':>7s} {len(doe_questions):>7d}")

The multiplier effect is clear: 130 base questions × 4 runs = 520 DOE questions, each with a unique factor assignment. This is the power of DOE-based testing — from a single ingested document we generate a systematic, balanced test suite that probes multiple dimensions of agent behavior simultaneously.

### Inspecting a DOE Question

Each `DOEQuestion` wraps the rephrased question, the factor assignment, and links back to its base question:

In [ ]:
# Inspect a single DOE question
dq = doe_questions[0]
print("=== DOE Question Anatomy ===")
print(f"  doe_qid:          {dq.doe_qid}")
print(f"  base_qid:         {dq.base_qid}")
print(f"  factor_assignment: {dq.factor_assignment}")
print("\n  Wrapped question:")
print(f"    category:        {dq.question.category}")
print(f"    natural_language: {dq.question.natural_language[:120]}")
print(f"    ground_truth:    {dq.question.ground_truth}")
print(f"    answer_type:     {dq.question.answer_type}")

---

## Exercises

### Exercise 5.1

Run the `PlausibilityGenerator` with the full candidate pool (no sampling). Plot a histogram of the geodesic margin (`geodesic_wrong - geodesic_real`) for all 68 questions. What is the median margin? Are there any questions with margins below 0.2 radians?

### Exercise 5.2

Use `generate_base_questions()` with the `comprehensive` factor group (20 factors) and `n_runs=8`. How many DOE questions does this produce? What is the ratio of DOE questions to base questions? Compute the factor coverage using `dm.coverage_report()`.

### Exercise 5.3

Write a function that takes a list of DOE questions and returns a DataFrame with columns `[doe_qid, category, clarity, entity_aliasing, reasoning_cue, ground_truth]`. Use it to verify that every (category, factor_assignment) combination appears at most once in the design.

In [ ]:
# === Exercise 5.1: Your code here ===
# Starter: uncomment and run
# plaus_gen = PlausibilityGenerator()
# all_plaus = plaus_gen.generate(store)
# margins = [q.metadata["geodesic_wrong"] - q.metadata["geodesic_real"] for q in all_plaus]
# import matplotlib.pyplot as plt
# plt.hist(margins, bins=20)
# plt.xlabel("Geodesic margin (radians)")
# plt.title(f"Plausibility margins (n={len(margins)}, median={np.median(margins):.3f})")
# plt.show()

In [ ]:
# === Exercise 5.2: Your code here ===


In [ ]:
# === Exercise 5.3: Your code here ===


---

## Summary

In this chapter we explored the question generation pipeline that turns a GMS expert store into a systematic test suite:

1. **Eight geometric generators** mine different GMS primitives (geodesic scoring, tension energy, holonomy, link prediction, multi-hop transport, phase comparison, cross-relation conditional and composed procedural transport) to produce 334 questions with provable ground truth — 100% pass self-validation. Together they cover the four fundamental query types: factual, relational, conditional and procedural.
2. **Five taxonomy generators** add failure-mode coverage — boundary, adversarial, robustness, recovery and governance — for a total of 13 non-redundant generators across 13 categories.
3. **The bridge** adapts eval-format questions to DOE-format questions via a closure that captures the store reference.
4. **The factor catalog** (40 factors in 10 families, 13 preset groups) defines how questions are systematically varied. The `quick_screen` group uses 3 factors (clarity, entity aliasing, reasoning cue) for lightweight screening.
5. **Two rephrasing methods** — template (deterministic, free, reproducible) and LLM (natural adversarial phrasing, non-deterministic) — transform base questions according to factor assignments while preserving ground truth.
6. **DOE expansion** multiplies 130 base questions × 4 Sobol-sampled conditions = 520 DOE questions, each with a unique factor assignment. The ground truth is preserved across all conditions.
7. **The multiplier effect** means a single ingested document yields hundreds of test cases that systematically probe different dimensions of agent behavior.

**Next:** Chapter 6 shows how the GMS geometric judge scores agent responses against these ground truths — turning raw answers into pass/fail verdicts with continuous confidence scores.